In [1]:
from aiida import load_profile, orm
from aiida.engine import submit, run
from aiida_pythonjob import PythonJob
from pymatgen.io.cif import CifParser
import numpy as np

load_profile()

Profile<uuid='dd482d6b5e0344d1a9ba01ea32d227fe' name='default'>

## 1. Load and Visualize Structure

Let's start by loading a simple structure (Silicon).

In [2]:
# Load structure from CIF
parser = CifParser("/home/jovyan/bind_mount/codes/aiida-muon/examples/data/LaCoPO_bcs_file_26735.mcif")
parser = CifParser("/home/jovyan/bind_mount/codes/aiida-muon/examples/data/Si.cif")
parser = CifParser("/home/jovyan/bind_mount/work/MLIPs_PROJECT/La6Ni4O14.cif")
structure_pmg = parser.get_structures(primitive=True)[0]
structure = orm.StructureData(pymatgen=structure_pmg)

print(f"Formula: {structure_pmg.formula}")
print(f"Number of atoms: {len(structure_pmg)}")
print(f"Lattice parameters: {structure_pmg.lattice.abc}")
print(f"Space group: {structure_pmg.get_space_group_info()}")

/tmp/ipykernel_101074/4190350754.py:5: FutureWarning: get_structures is deprecated; use parse_structures in pymatgen.io.cif instead.
The only difference is that primitive defaults to False in the new parse_structures method.So parse_structures(primitive=True) is equivalent to the old behavior of get_structures().
  structure_pmg = parser.get_structures(primitive=True)[0]


Formula: La6 Ni4 O14
Number of atoms: 24
Lattice parameters: (3.7611, 3.7611, 19.8582)
Space group: ('P4/mmm', 123)


## 2. Setup MLIP Calculator

We'll use MACE-MP (Materials Project) as our MLIP calculator. The calculator must be defined as a function that will be executed remotely.

In [3]:
def get_mace_calculator():
    """Create and return a MACE calculator.
    
    This function will be executed on the remote computer.
    All imports must be inside the function.
    """
    from mace.calculators import mace_mp
    return mace_mp(
        model="medium",      # Options: small, medium, large
        device="cpu",        # Use 'cuda' if GPU available
        default_dtype="float64",
        dispersion=False     # Set True to include dispersion corrections
    )

def get_mattersim_calculator():
    """Create and return a Mattersim calculator.
    
    This function will be executed on the remote computer.
    All imports must be inside the function.
    
    Note: Works around pkg_resources import in mattersim.__version__.
    """
    import sys
    
    # Workaround for missing pkg_resources: patch mattersim.__version__ module
    # This prevents the import error when mattersim tries to import pkg_resources
    class FakeVersion:
        __version__ = "1.0.0"  # Dummy version
    
    # Pre-create the __version__ module to avoid pkg_resources import
    import types
    version_module = types.ModuleType('mattersim.__version__')
    version_module.__version__ = "1.0.0"
    sys.modules['mattersim.__version__'] = version_module
    
    from mattersim.forcefield import MatterSimCalculator
    mattersim_calculator = MatterSimCalculator(
        #load_path="/home/jovyan/bind_mount/codes/mattersim/pretrained_models/mattersim-v1.0.0-5M.pth",
        load_path="/home/bonacc_m/Codes/mattersim/pretrained_models/mattersim-v1.0.0-5M.pth",
        device="cpu",
    )
    return mattersim_calculator

def get_nequip_calculator():
    """Create and return a NEquIP calculator.
    
    This function will be executed on the remote computer.
    All imports must be inside the function.
    
    Note: Handles PyTorch 2.6 weights_only compatibility issue with e3nn.
    """
    # Fix for PyTorch 2.6: Allow slice in safe globals for e3nn's constants.pt
    import torch
    torch.serialization.add_safe_globals([slice])
    
    from nequip.ase import NequIPCalculator
    nequip_calculator = NequIPCalculator.from_compiled_model(
        compile_path="/home/jovyan/bind_mount/codes/compiled_mp_l_01.nequip.pt2",
        #compile_path="/home/bonacc_m/Codes/compiled_mp_l_01.nequip.pt2",
        device="cpu",
    )
    return nequip_calculator


In [4]:
mlips = ['mace', 'mattersim', 'nequip']

mlip = mlips[1] 

# Load your configured PythonJob code
# If not configured, run: verdi code create core.code.installed ...
if mlip == 'mace':
    pythonjob_code = orm.load_code('python3@localhost')
    callback_calculator = get_mace_calculator
elif mlip == 'mattersim':
    pythonjob_code = orm.load_code('python3_mattersim_p311@mpc3129')
    callback_calculator = get_mattersim_calculator
elif mlip == 'nequip':
    pythonjob_code = orm.load_code('python3_nequip_p311@mpc3129')
    callback_calculator = get_nequip_calculator

pythonjob_metadata = {
    'label': f'{mlip}_relaxation',
    'description': f'relaxation using {mlip} calculator',
    'options': {
        'resources': {'num_machines': 1, 'num_mpiprocs_per_machine': 1},
        'max_wallclock_seconds': 3600,
    }
}

## 3. Prepare Inputs



In [5]:
from aiida_muon.workflows.find_muon import FindMuonWorkChain

In [6]:
# Supercell Matrix and other relevant inputs.
sc_matrix = [[3, 0, 0], [0, 3, 0], [0, 0, 1]]
mu_spacing = 1.0
kpoints_distance = 0.3
charge_supercell = False

# Codes.
codename = "pw-7.3@thor"  # edit 
code = orm.load_code(codename)
pp_codename = "pp-7.3@thor"  # edit 
pp_code = orm.load_code(pp_codename)
# Resources dictionary with a minimal computational settings.
resources = {    
    "num_machines": 1,
    "num_mpiprocs_per_machine": 48,
    "num_cores_per_mpiproc": 1,
}


In [7]:
# NOTE: if we do supercells_list, the sc_matrix and mu_spacing are not used.

builder = FindMuonWorkChain.get_builder_from_protocol(
    pw_code=code,
    pp_code = pp_code,
    structure = structure, #orm.load_node(35926), # orm.load_node(21259), # <- LiF
    #magmom = magmom,
    hubbard=False,
    #hubbard_dict = {"Mn":5},
    sc_matrix = sc_matrix,
    mu_spacing = mu_spacing,
    kpoints_distance = 0.3, # I want Gamma
    charge_supercell = charge_supercell,
    gamma_pre_relax = False,
    full_dft_relax = False, # if True, will skip DFT relaxations and directly run impurity supercell conversions and post-processing on the initial structure. Default is False.
   
    pseudo_family = "SSSP/1.3/PBE/efficiency",

    ML_pre_relax = True, # if True, will run ML relaxations before DFT relaxations. Default is False.
    pythonjob_code = pythonjob_code, # if ML_pre_relax is True, this PythonJob code will be used to run the ML relaxations. Default is None.      
    callback_calculator = callback_calculator, # if ML_pre_relax is True, this callback function will be called to get the ML calculator for pre-relaxation. Default is None.

    #overrides = overrides,
    #supercells_list = [supercell_02.uuid], #[Gamma_relaxed_structure.uuid],
    #enforce_defaults = True, # if False, will use kpoints and threhsolds from aiida-quantumespresso protocols. Default is True.
    )

builder.pwscf.pw.metadata.options.resources = resources
builder.pwscf.pw.metadata.options.prepend_text = "export OMP_NUM_THREADS=1"

builder.relax.base.pw.metadata.options.resources = resources
builder.relax.base.pw.metadata.options.prepend_text = "export OMP_NUM_THREADS=1"
builder.relax.base.pw.metadata.options.max_wallclock_seconds = 24*60*60

builder.relax.base.pw.parallelization = orm.Dict({'npool': 1})

builder.impuritysupercellconv_metadata = {"options":{
    'resources':resources,
    'prepend_text':"export OMP_NUM_THREADS=1",
    },}

builder.pp_metadata = {"options":{
    'resources':resources,
    'prepend_text':"export OMP_NUM_THREADS=1",
    'max_wallclock_seconds': 24*60*60, # 1h
    },}

/home/jovyan/.conda/envs/base_311/lib/python3.11/site-packages/aiida_quantumespresso/workflows/protocols/utils.py:147: UserWarning: Found unrecognised key in overrides: base.pseudo_family
  warnings.warn(f'Found unrecognised key in overrides: {full_key}')
/home/jovyan/.conda/envs/base_311/lib/python3.11/site-packages/aiida_quantumespresso/workflows/protocols/utils.py:147: UserWarning: Found unrecognised key in overrides: base_final_scf.pseudo_family
  warnings.warn(f'Found unrecognised key in overrides: {full_key}')
/home/jovyan/.conda/envs/base_311/lib/python3.11/site-packages/aiida_quantumespresso/workflows/protocols/utils.py:147: UserWarning: Found unrecognised key in overrides: base.pseudo_family
  warnings.warn(f'Found unrecognised key in overrides: {full_key}')
/home/jovyan/.conda/envs/base_311/lib/python3.11/site-packages/aiida_quantumespresso/workflows/protocols/utils.py:147: UserWarning: Found unrecognised key in overrides: base_final_scf.pseudo_family
  warnings.warn(f'Found 

In [8]:
from aiida.engine import submit

In [9]:
run = submit(builder)

In [10]:
run.pk

81019

In [11]:
run.label = f'{mlip}_relaxation'